# MindRL Challenge — Data Loading

This notebook loads the MindRL Challenge public dataset and transforms it into the
trial-level DataFrame format expected by **HSSM** / `ssms.rl`.

**Task:** 4-arm drifting (restless) bandit — participants choose one of 4 actions per
trial and receive scalar reward feedback. Rewards drift over time, so participants
must balance learning from history against tracking recent changes.

**Data source:** [HuggingFace](https://huggingface.co/datasets/mindrl-hub/mindrl-challenge-public)

## 1. Setup

Run `uv sync` first to install all dependencies (HSSM, ssms, huggingface_hub, etc.).
The data is downloaded automatically in the next cell if not already present.

In [ ]:
import logging
import warnings

import pandas as pd

from bayesd_misfits.data import (
    ensure_data_downloaded,
    load_challenge_data,
    summarize_data,
    validate_for_hssm,
)

warnings.filterwarnings("ignore")
logging.getLogger("jax._src.xla_bridge").setLevel("ERROR")

## 2. Download data (if not already present)

Fetches the dataset from HuggingFace (~40 MB) on first run. Skips download if files
already exist in `hf_cache/public/`.

In [ ]:
ensure_data_downloaded()

## 3. Load data

We load the JSONL trajectories and flatten them into a trial-level DataFrame.

Key options:
- **`feedback_transform`**: How to convert reward (1–100) into the `feedback` column
  that the RL learner uses to update Q-values.
  - `"normalize"` → `reward / 100` (scales to [0, 1])
  - `"binary"`    → threshold at `binary_threshold` (default 50) → 0 or 1
  - `"raw"`       → keep original 1–100 scale
- **`group_by`**: Whether `participant_id` maps to subject or trajectory.
  - `"trajectory"` — each trajectory is a separate participant (balanced 108–120
    trial panels; recommended for RLSSM where Q-values reset per block).
  - `"subject"`    — per-subject grouping with continuous `trial_id` across blocks.
- **`rt_placeholder`**: Value for missing RTs. Use `-1.0` for choice-only models,
  or set `drop_missing_rt=True` to remove them for RT-based models.

In [ ]:
# ── Choice-only RLSSM (recommended starting point) ──
# Feedback normalized to [0, 1], RT set to -1.0 placeholder, one trajectory = one participant.
df = load_challenge_data(
    feedback_transform="normalize",
    rt_placeholder=-1.0,
    group_by="trajectory",
)

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(10)

### HSSM-ready columns

| Column | Type | Description |
|---|---|---|
| `participant_id` | int | Subject or trajectory ID (for hierarchical random effects) |
| `trial_id` | int | Trial index (continuous within participant) |
| `response` | int | Chosen action (0–3) |
| `rt` | float | Response time in seconds (`-1.0` = placeholder for choice-only) |
| `feedback` | float | Reward received (used by RL learner to update Q-values) |
| `global_trial_id` | int | Original within-trajectory trial index (for schedule cross-ref) |
| `subject_id` | str | Raw subject identifier (e.g. `"sub_000848"`) |
| `trajectory_id` | str | Raw trajectory identifier (e.g. `"traj_005472"`) |
| `block_id` | str | Block identifier |
| `reward_raw` | float | Original reward before transform (1–100) |

## 4. Summary & validation

In [ ]:
summarize_data(df)

In [ ]:
validate_for_hssm(df, require_rt=False)

## 5. Alternative configurations

Uncomment the cell that matches your modeling approach.

In [ ]:
# ── RT-based RLSSM (choice + response time) ──
# Drops trials with missing RTs, keeps raw reward scale.
# df_rt = load_challenge_data(
#     feedback_transform="normalize",
#     drop_missing_rt=True,
#     rt_placeholder=None,
#     group_by="trajectory",
# )
# df_rt.head()

In [ ]:
# ── Per-subject grouping (continuous trial_id across blocks) ──
# Use when hierarchical random effects should be per-subject, not per-trajectory.
# Note: Q-values will carry over between a subject's trajectories.
# df_subj = load_challenge_data(
#     feedback_transform="normalize",
#     rt_placeholder=-1.0,
#     group_by="subject",
# )
# df_subj.head()

In [ ]:
# ── With reward schedule sidecar (for analysis / generative simulation) ──
# Exposes all 4 arms' rewards per trial (including unchosen).
# ⚠️ If used in your model, disclose it in your Interpretation Card.
# df, schedules = load_challenge_data(
#     feedback_transform="normalize",
#     rt_placeholder=-1.0,
#     group_by="trajectory",
#     include_reward_schedules=True,
# )
# schedules.head()

## 6. Prepare minimal HSSM data

For choice-only RLSSM, HSSM needs only `response`, `participant_id`, `trial_id`,
and `feedback`. The `rt` column can be dropped (or kept as `-1.0` placeholder
depending on the model preset).

In [ ]:
# Minimal columns for choice-only HSSM RLSSM
hssm_data = df[["participant_id", "trial_id", "response", "feedback"]].copy()

print(f"Shape: {hssm_data.shape}")
hssm_data.head(10)

In [ ]:
# For RT-based RLSSM, include the rt column (after dropping missing RTs)
# hssm_data_rt = load_challenge_data(
#     feedback_transform="normalize",
#     drop_missing_rt=True,
#     group_by="trajectory",
# )[["participant_id", "trial_id", "response", "rt", "feedback"]]
# hssm_data_rt.head()